# Transformers Causal Decoder 

this is the baseline mechanism of GPT-2 and GPT-3 

using SGD, we first get batched input -> token embedding + positioning embedding -> transformers block-> estimate loss

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-11 04:31:44--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  84.9KB/s    in 8.6s    

2026-10-11 04:31:53 (127 KB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass

# read data
with open('input.txt') as f:
    text = f.read()

# activate cuda
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
if device == 'cuda':
    torch.cuda.manual_seed(1337)


# configuration
@dataclass
class TCDEconfig:
    block_size: int = 32
    batch_size: int = 16
    n_layer: int = 4
    n_embd: int = 64
    n_head: int = 4
    dropout: float = 0.0
    lr: float = 1e-3
    test_interval: int = 100
    epochs: int = 5000
    test_epochs: int = 200


config = TCDEconfig()  # changeable, e.g TCDEconfig(n_embd=128, dropout=0.3)

# set vocab and vocab_size
vocab = sorted(list(set(text)))
vocab_size = len(vocab)

# tokenization
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda l: ''.join([itos[i] for i in l])
data = torch.tensor(encoder(text), dtype=torch.long)  # list -> tensor

# split data
holdout = int(0.9 * len(data))
train_data = data[:holdout]
test_data = data[holdout:]


# get_batch data
def get_batch(config, split):
    data_used = train_data if split == 'train' else test_data
    initial_index = torch.randint(len(data_used) - config.block_size, (config.batch_size,))  # B
    x = torch.stack([data_used[ix: ix + config.block_size] for ix in initial_index])  # B T
    y = torch.stack([data_used[ix + 1: ix + config.block_size + 1] for ix in initial_index])  # B T
    x = x.to(device)
    y = y.to(device)
    return x, y


# language model
class transformers(nn.Module):  # one attention head
    def __init__(self, config, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(config.n_embd, head_size, bias=False)
        self.query = nn.Linear(config.n_embd, head_size, bias=False)
        self.value = nn.Linear(config.n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)
        self.register_buffer('tril', torch.tril(torch.ones([config.block_size, config.block_size])))

    def forward(self, x):
        B, T, C = x.shape
        K = self.key(x)
        Q = self.query(x)
        att = Q @ K.transpose(-2, -1) * self.head_size ** -0.5
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = nn.functional.softmax(att, dim=-1)
        att = self.dropout(att)

        V = self.value(x)
        out = att @ V
        return out


class multiheadattention(nn.Module):
    def __init__(self, config, head_size):
        super().__init__()
        self.head = nn.ModuleList([transformers(config, head_size) for _ in range(config.n_head)])
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = torch.cat([h(x) for h in self.head], dim=-1)
        return self.dropout(self.proj(x))


class ffn(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.ReLU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout))

    def forward(self, x):
        return self.network(x)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.mulhead = multiheadattention(config, head_size)
        self.net = ffn(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.mulhead(self.ln1(x))
        x = x + self.net(self.ln2(x))
        return x


class LanguageModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.block_size = config.block_size
        self.token_embedding_table = nn.Embedding(vocab_size, config.n_embd)
        self.pos_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.block = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.final_lm = nn.Linear(config.n_embd, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        toke_embd = self.token_embedding_table(x)                      # B T C
        pos_embd = self.pos_embedding_table(torch.arange(T, device=x.device))  # T C
        X = toke_embd + pos_embd
        X = self.block(X)                                              # pass through all blocks
        logits = self.final_lm(self.ln_final(X))                       # B T V

        # if we want to generate text, no need to use targets, so targets=None
        if targets is None:
            return logits, None

        B, T, V = logits.shape
        logits = logits.view(B * T, V)
        targets = targets.view(B * T)
        loss = nn.functional.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):  # idx = input_batch
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]       # crop context
            logits, _ = self(idx_cond)                 # B T V
            logits = logits[:, -1, :]                  # B V (last token only)
            probs = nn.functional.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# estimate loss and accuracies
@torch.no_grad()
def estimate_loss_accuracy(config, m):
    output = {}
    m.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(config.test_epochs)
        accs = torch.zeros(config.test_epochs)
        for j in range(config.test_epochs):
            x, y = get_batch(config, split)
            logits, loss = m(x, y)               # logits: (B*T, V)
            pred = torch.argmax(logits, dim=-1)  # (B*T)
            y = y.reshape(-1)                    # (B*T)
            losses[j] = loss.item()
            accs[j] = (pred == y).float().mean() * 100
        output[split] = [accs.mean().item(), losses.mean().item()]  # [acc, loss]
    m.train()
    return output


model = LanguageModel(config)
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)


def train_model(config):
    for iters in range(config.epochs):
        if iters % config.test_interval == 0 or iters == config.epochs - 1:
            output = estimate_loss_accuracy(config, m)
            tr_acc, tr_loss = output['train']
            te_acc, te_loss = output['test']
            print(f'epochs: {iters+1}, train_acc = {tr_acc:.2f}, train_loss = {tr_loss:.4f}, '
                  f'test_acc = {te_acc:.2f}, test_loss = {te_loss:.4f}')

        optimizer.zero_grad()
        x, y = get_batch(config, 'train')
        logits, loss = m(x, y)
        loss.backward()
        optimizer.step()


train_model(config)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decoder(m.generate(context, max_new_tokens=2000)[0].tolist()))